<a href="https://colab.research.google.com/github/CavalierTomWard/SMS-CL_automations/blob/main/SMSB%26R%2C_SMS_MMS_stripping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import re
import os

# 1. Mount Google Drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/SMSB&RP'
sms_xml_path = os.path.join(folder_path, 'sms-2026-10-28_00-56-16.xml')
output_csv = os.path.join(folder_path, 'sms_text_only_2026-10-28.csv')

# 2. Normalization Function
def normalize_phone(num):
    if pd.isna(num):
        return 'Unknown'
    s = str(num).strip()
    if any(c.isalpha() for c in s):
        return s
    cleaned = re.sub(r'[^\d+]', '', s)
    if not cleaned:
        return s
    if cleaned.startswith('0') and len(cleaned) == 10:
        cleaned = '+61' + cleaned[1:]
    elif len(cleaned) == 9 and cleaned.startswith('4'):
        cleaned = '+61' + cleaned
    return cleaned

print("Streaming and parsing SMS & MMS XML backup...")
records = []

# 3. Stream parse both <sms> and <mms> tags
for event, elem in ET.iterparse(sms_xml_path, events=('end',)):
    if elem.tag == 'sms':
        records.append({
            'readable_date': elem.attrib.get('readable_date', ''),
            'address': elem.attrib.get('address', ''),
            'contact_name': elem.attrib.get('contact_name', '(Unknown)'),
            'type': 'Received' if elem.attrib.get('type') == '1' else 'Sent',
            'body': elem.attrib.get('body', '')
        })
        elem.clear()
    elif elem.tag == 'mms':
        # mms msg_box: 1 = Received, 2 = Sent
        box = elem.attrib.get('msg_box', '1')
        msg_type = 'Received' if box == '1' else 'Sent'
        date_ms = elem.attrib.get('date', '0')
        readable_date = elem.attrib.get('readable_date', '')

        # Extract text parts from MMS
        body_text = ""
        address = elem.attrib.get('address', '')

        for sub in elem.iter():
            if sub.tag == 'part' and sub.attrib.get('ct') == 'text/plain':
                body_text += sub.attrib.get('text', '') + " "

        if body_text.strip():
            records.append({
                'readable_date': readable_date,
                'address': address,
                'contact_name': elem.attrib.get('contact_name', '(Unknown)'),
                'type': msg_type,
                'body': body_text.strip()
            })
        elem.clear()

# 4. Export to CSV
df_sms = pd.DataFrame(records)
df_sms.to_csv(output_csv, index=False, encoding='utf-8')
print(f"Successfully extracted {len(df_sms)} total messages (SMS + MMS) to {output_csv}")

Mounted at /content/drive
Streaming and parsing SMS & MMS XML backup...
Successfully extracted 36627 total messages (SMS + MMS) to /content/drive/MyDrive/SMSB&RP/sms_text_only_2026-10-28.csv
